# Mixed-effects models

Session-level averages lose the trial-by-trial structure of the data. Here each trial is a separate row, and the models account for the fact that trials are nested within users - everyone has their own baseline, and everyone has their own drift across the session.

Depression scores are z-standardized (PHQ-9 and BDI-II separately), so coefficients are in units of "one standard deviation of depression." Trial position is rescaled to 0–1 per session.

Each model tries a random intercept and random slope per user. If the slope variance collapses toward zero, the model is refit with random intercept only. The `random_structure` column records which one was used.

For each fit the summary reports the coefficient, raw p-value, Benjamini-Hochberg FDR-corrected p-value, a pseudo-Cohen's d (coefficient divided by the square root of between-person plus residual variance), and the ICC (fraction of variance that is between-person rather than trial-to-trial noise).

PHQ-9 and BDI-II are run independently and compared at the end.

## Depression х trial position

For each gaze metric we fit:

`metric ~ depression_score * trial_norm`

This asks two questions. First, does the metric differ between more and less depressed users on average (main effect)? Second, does the within-session drift depend on depression level (interaction)?

In [0]:
%pip install statsmodels -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

from src.evaluation.lmm_temporal import fit_all, apply_fdr
from src.mixed_effects import plot_trajectories

## 1. Load data

In [0]:
METRICS = [
    "fixation_bias",
    "fixation_proportion_negative", "fixation_proportion_positive", "fixation_proportion_neutral",
    "dwell_time_ms_negative", "dwell_time_ms_positive", "dwell_time_ms_neutral",
    "dwell_time_500ms_negative", "dwell_time_500ms_positive", "dwell_time_500ms_neutral",
    "first_fixation_duration_ms", "second_fixation_duration_ms",
    "ttff_ms_negative", "ttff_ms_positive", "ttff_ms_neutral",
    "revisit_count_negative", "revisit_count_positive",
    "mean_fixation_duration_ms", "fixation_rate_per_sec",
    "mean_saccade_amplitude", "saccade_rate_per_sec",
    "scanpath_length", "gaze_transition_entropy", "transition_matrix_density",
    "blink_rate_per_min",
]

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))

cols = ["session_id", "scene_index", "trial_num"] + METRICS

df = numbered.select(*cols).join(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score"),
    on="session_id", how="inner",
).toPandas()

df["trial_norm"] = df.groupby("session_id")["trial_num"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0
)
df["phq9_z"] = (df["phq9_score"] - df["phq9_score"].mean()) / df["phq9_score"].std()
df["bdi_z"] = (df["bdi_score"] - df["bdi_score"].mean()) / df["bdi_score"].std()

df[["session_id", "uid", "phq9_score", "bdi_score"]].nunique()

## 2. PHQ-9

### 2.1 Fit all models

In [0]:
phq_results, phq_summary = fit_all(df, METRICS, "phq9_z")
phq_summary = apply_fdr(phq_summary, "phq9_z")
phq_summary

### 2.2 FDR-significant effects (α = 0.05)

In [0]:
phq_summary[phq_summary["phq9_z_pval_fdr"] < 0.05][
    ["metric", "phq9_z_coef", "phq9_z_pval_fdr", "phq9_z_d", "icc"]
].sort_values("phq9_z_pval_fdr")

In [0]:
phq_summary[phq_summary["phq9_z:trial_norm_pval_fdr"] < 0.05][
    ["metric", "phq9_z:trial_norm_coef", "phq9_z:trial_norm_pval_fdr"]
]

### 2.3 Trajectories

In [0]:
plot_trajectories(df, phq_summary, "phq9_z", "PHQ-9", "phq9_score", METRICS)

## 3. BDI-II

### 3.1 Fit all models

In [0]:
bdi_results, bdi_summary = fit_all(df, METRICS, "bdi_z")
bdi_summary = apply_fdr(bdi_summary, "bdi_z")
bdi_summary

### 3.2 FDR-significant effects (α = 0.05)

In [0]:
bdi_summary[bdi_summary["bdi_z_pval_fdr"] < 0.05][
    ["metric", "bdi_z_coef", "bdi_z_pval_fdr", "bdi_z_d", "icc"]
].sort_values("bdi_z_pval_fdr")

In [0]:
bdi_summary[bdi_summary["bdi_z:trial_norm_pval_fdr"] < 0.05][
    ["metric", "bdi_z:trial_norm_coef", "bdi_z:trial_norm_pval_fdr"]
]

### 3.3 Trajectories

In [0]:
plot_trajectories(df, bdi_summary, "bdi_z", "BDI", "bdi_score", METRICS)

## 4. Detailed fits for the key metrics

In [0]:
KEY_METRICS = [
    "fixation_bias",
    "dwell_time_ms_negative", "dwell_time_ms_positive",
    "fixation_proportion_negative", "fixation_proportion_positive",
]

for metric in KEY_METRICS:
    print(f"\n=== {metric} (PHQ-9) ===")
    print(phq_results[metric].summary())
    print(f"\n=== {metric} (BDI-II) ===")
    print(bdi_results[metric].summary())

## 5. PHQ-9 vs BDI-II comparison

In [0]:
comparison = phq_summary[["metric", "phq9_z_coef", "phq9_z_pval_fdr", "phq9_z_d", "icc"]].merge(
    bdi_summary[["metric", "bdi_z_coef", "bdi_z_pval_fdr", "bdi_z_d"]],
    on="metric",
).sort_values("phq9_z_pval_fdr")

comparison

In [0]:
sig_phq = comparison["phq9_z_pval_fdr"] < 0.05
sig_bdi = comparison["bdi_z_pval_fdr"] < 0.05

pd.Series({
    "both":      (sig_phq & sig_bdi).sum(),
    "phq_only":  (sig_phq & ~sig_bdi).sum(),
    "bdi_only":  (~sig_phq & sig_bdi).sum(),
    "neither":   (~sig_phq & ~sig_bdi).sum(),
})

## 6. Reliability (ICC)

In [0]:
phq_summary[["metric", "icc"]].sort_values("icc", ascending=False)

In [0]:
phq_summary["icc"].agg(["mean", "median"]).round(3)